# NBGrader - Sistema Automatizado de Calificación

## Observaciones Importantes:
1. Se debe reiniciar la sesión si se copian archivos nuevos de estudiantes
2. Los nombres de carpetas NO pueden tener acentos (el feedback no funciona)
3. Cargar estudiantes desde CSV: `estudiantes.csv`

## Configuración
- **Curso:** Python_AP
- **Assignment:** S01_D02_A02
- **Directorio base:** /content/drive/MyDrive/nbgrader_PAGD1_14123

## 1. Instalación de Dependencias

In [ ]:
# Instalar nbclient 0.6.1
!pip install nbclient==0.6.1 -q

In [ ]:
# Instalar nbgrader 0.8.1
!pip install nbgrader==0.8.1 -q

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración del Curso

In [ ]:
import os
import pandas as pd
import shutil
from pathlib import Path

# Configuración global
BASE_PATH = '/content/drive/MyDrive/nbgrader_PAGD1_14123'
ASSIGNMENT_ID = 'S01_D02_A02'
COURSE_ID = 'Python_AP'

# Rutas de directorios
DIRS = {
    'source': os.path.join(BASE_PATH, 'source'),
    'release': os.path.join(BASE_PATH, 'release'),
    'submitted': os.path.join(BASE_PATH, 'submitted'),
    'autograded': os.path.join(BASE_PATH, 'autograded'),
    'feedback': os.path.join(BASE_PATH, 'feedback')
}

print(f"Base Path: {BASE_PATH}")
print(f"Assignment: {ASSIGNMENT_ID}")
print(f"Course: {COURSE_ID}")

## 4. Crear Estructura de Directorios

In [ ]:
# Crear directorios base del curso
for dir_name, dir_path in DIRS.items():
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"✓ Directorio '{dir_name}' OK")
    except OSError as error:
        print(f"✗ Error creando '{dir_name}': {error}")

## 5. Cargar Lista de Estudiantes

### Formato del CSV esperado:
```
nombre_estudiante
Apellido1_Apellido2_Nombre1_Nombre2
...
```

**IMPORTANTE:** Los nombres NO deben tener acentos ni caracteres especiales.

In [ ]:
from google.colab import files

# Opción 1: Subir archivo CSV
print("Por favor, sube el archivo estudiantes.csv")
uploaded = files.upload()

# Leer CSV
csv_filename = list(uploaded.keys())[0]
df_estudiantes = pd.read_csv(csv_filename)

# Validar que existe la columna requerida
if 'nombre_estudiante' not in df_estudiantes.columns:
    raise ValueError("El CSV debe tener una columna llamada 'nombre_estudiante'")

# Obtener lista de estudiantes
estudiantes = df_estudiantes['nombre_estudiante'].tolist()

# Validar nombres (sin acentos)
import re
for est in estudiantes:
    if not re.match(r'^[a-zA-Z0-9_]+$', est):
        print(f"⚠️  ADVERTENCIA: '{est}' contiene caracteres especiales o acentos")

print(f"\n✓ Se cargaron {len(estudiantes)} estudiantes:")
for i, est in enumerate(estudiantes, 1):
    print(f"  {i}. {est}")

### Opción alternativa: Definir estudiantes manualmente

In [ ]:
# Descomentar si prefieres definir la lista manualmente
# estudiantes = [
#     'Acosta_Zavaleta_Jose_Manuel',
#     'Alza_Guzman_Andrea_Coral',
#     'Asencios_Clavijo_Gabriela_Shumey',
#     'Claros_Condori_Marife_Mardely',
#     'Enriquez_Martinez_Romina_Sofia',
#     'Huayhuapuma_De_La_Cruz_Anais',
#     'Medina_Mendieta_Maria_Fernanda_Valentina',
#     'Rosales_Alarcon_Alexis_Andre',
#     'Silva_Arzapalo_Rafael',
#     'Terreros_Gutarra_Yamile_Milagros'
# ]
# print(f"✓ Lista manual: {len(estudiantes)} estudiantes")

## 6. Crear Carpetas de Estudiantes

In [ ]:
# Crear carpetas para cada estudiante en 'submitted'
submitted_base = DIRS['submitted']

for estudiante in estudiantes:
    # Crear carpeta del estudiante
    student_path = os.path.join(submitted_base, estudiante)
    os.makedirs(student_path, exist_ok=True)
    
    # Crear subcarpeta del assignment
    assignment_path = os.path.join(student_path, ASSIGNMENT_ID)
    os.makedirs(assignment_path, exist_ok=True)
    
    print(f"✓ {estudiante}/{ASSIGNMENT_ID}")

print(f"\n✓ Carpetas creadas para {len(estudiantes)} estudiantes")

## 7. Generar Assignment para Estudiantes

**Prerequisito:** Debes tener el notebook fuente en `source/{ASSIGNMENT_ID}/{ASSIGNMENT_ID}.ipynb`

In [ ]:
# Generar assignment
!nbgrader generate_assignment --assignment_id='{ASSIGNMENT_ID}' --debug

## 8. Autograding Masivo

**Prerequisito:** Los estudiantes deben haber subido sus notebooks en `submitted/{estudiante}/{ASSIGNMENT_ID}/`

In [ ]:
# Autograding para todos los estudiantes
import subprocess

errores = []
exitosos = []

for i, estudiante in enumerate(estudiantes, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(estudiantes)}] Calificando: {estudiante}")
    print(f"{'='*60}")
    
    try:
        cmd = f"nbgrader autograde --student {estudiante} {ASSIGNMENT_ID} --debug"
        result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
        
        if result.returncode == 0:
            exitosos.append(estudiante)
            print(f"✓ {estudiante} - OK")
        else:
            errores.append(estudiante)
            print(f"✗ {estudiante} - ERROR")
    except Exception as e:
        errores.append(estudiante)
        print(f"✗ {estudiante} - EXCEPTION: {e}")

print(f"\n{'='*60}")
print("RESUMEN DE CALIFICACIÓN")
print(f"{'='*60}")
print(f"✓ Exitosos: {len(exitosos)}/{len(estudiantes)}")
print(f"✗ Errores: {len(errores)}/{len(estudiantes)}")

if errores:
    print(f"\nEstudiantes con errores:")
    for est in errores:
        print(f"  - {est}")

## 9. Generar Feedback Masivo

In [ ]:
# Generar feedback para todos los estudiantes
!nbgrader feedback --assignment {ASSIGNMENT_ID} --debug

## 10. Exportar Notas a CSV

In [ ]:
# Exportar calificaciones
OUTPUT_CSV = f"notas_{ASSIGNMENT_ID}.csv"
!nbgrader export --to {OUTPUT_CSV}

# Descargar archivo
from google.colab import files
files.download(OUTPUT_CSV)

## 11. Utilidades de Mantenimiento

### 11.1 Eliminar Submission de Estudiante(s)

In [ ]:
# Eliminar submission específica (para permitir re-submission)
estudiante_eliminar = "Acosta_Zavaleta_Jose_Manuel"  # Cambiar según necesidad

!nbgrader db student remove {estudiante_eliminar} --assignment {ASSIGNMENT_ID} --force

print(f"✓ Submission eliminada para: {estudiante_eliminar}")

### 11.2 Eliminar Múltiples Submissions

In [ ]:
# Eliminar submissions de múltiples estudiantes
estudiantes_eliminar = [
    # Descomentar los que necesites eliminar
    # 'Acosta_Zavaleta_Jose_Manuel',
    # 'Alza_Guzman_Andrea_Coral',
]

for est in estudiantes_eliminar:
    !nbgrader db student remove {est} --assignment {ASSIGNMENT_ID} --force
    print(f"✓ {est}")

print(f"\n✓ {len(estudiantes_eliminar)} submissions eliminadas")

### 11.3 Limpiar Directorio Feedback

In [ ]:
# Eliminar todo en feedback (pero no la carpeta)
feedback_path = DIRS['feedback']

for item in os.listdir(feedback_path):
    item_path = os.path.join(feedback_path, item)
    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
        print(f"✓ Eliminado: {item}")
    except Exception as e:
        print(f"✗ Error eliminando {item}: {e}")

print("\n✓ Directorio feedback limpiado")

### 11.4 Limpiar Directorio Autograded

In [ ]:
# Eliminar todo en autograded (pero no la carpeta)
autograded_path = DIRS['autograded']

for item in os.listdir(autograded_path):
    item_path = os.path.join(autograded_path, item)
    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
        print(f"✓ Eliminado: {item}")
    except Exception as e:
        print(f"✗ Error eliminando {item}: {e}")

print("\n✓ Directorio autograded limpiado")

### 11.5 Limpiar Directorio Release

In [ ]:
# Eliminar todo en release (pero no la carpeta)
release_path = DIRS['release']

for item in os.listdir(release_path):
    item_path = os.path.join(release_path, item)
    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
        print(f"✓ Eliminado: {item}")
    except Exception as e:
        print(f"✗ Error eliminando {item}: {e}")

print("\n✓ Directorio release limpiado")

### 11.6 Ver Estadísticas del Assignment

In [ ]:
# Verificar estado de submissions
submitted_base = DIRS['submitted']

print(f"Estado de submissions para {ASSIGNMENT_ID}:")
print(f"{'='*60}")

submissions_ok = []
submissions_faltantes = []

for estudiante in estudiantes:
    assignment_path = os.path.join(submitted_base, estudiante, ASSIGNMENT_ID)
    
    if os.path.exists(assignment_path):
        files_in_dir = os.listdir(assignment_path)
        nb_files = [f for f in files_in_dir if f.endswith('.ipynb')]
        
        if nb_files:
            submissions_ok.append(estudiante)
            print(f"✓ {estudiante}: {len(nb_files)} archivo(s)")
        else:
            submissions_faltantes.append(estudiante)
            print(f"⚠  {estudiante}: carpeta vacía")
    else:
        submissions_faltantes.append(estudiante)
        print(f"✗ {estudiante}: no existe carpeta")

print(f"\n{'='*60}")
print(f"RESUMEN")
print(f"{'='*60}")
print(f"Total estudiantes: {len(estudiantes)}")
print(f"✓ Con submissions: {len(submissions_ok)}")
print(f"✗ Sin submissions: {len(submissions_faltantes)}")

if submissions_faltantes:
    print(f"\nEstudiantes sin submission:")
    for est in submissions_faltantes:
        print(f"  - {est}")

## 12. Links de Referencia

### Para subir notebooks (estudiantes):
https://drive.google.com/drive/folders/1-62NTbmBBzWNxijWJ3D-2UvKoVwFnqFf?usp=sharing

### Para ver el feedback (estudiantes):
https://drive.google.com/drive/folders/1--qXKGDzyDiPlhDMonpB8jo9qvqswBlA?usp=sharing